# Studio Almasawa — real AI video generation\n\nThis notebook runs Wan2.1 T2V-1.3B on a Kaggle GPU. It creates a short generated video clip (level A), not a still-image animation.\n

In [ ]:
# PROMPT_CONFIG\nPROMPT = "__PROMPT__"\nWIDTH = 480\nHEIGHT = 832\nSEED = 42\nMODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"\n

In [ ]:
%pip -q install --upgrade diffusers transformers accelerate ftfy imageio[ffmpeg]\nimport json, os, subprocess, time\nfrom pathlib import Path\nimport torch\nfrom diffusers import WanPipeline\nfrom diffusers.utils import export_to_video\n\nif not torch.cuda.is_available():\n    raise RuntimeError("Kaggle GPU is not available. Enable GPU in the notebook settings and rerun.")\ngpu = torch.cuda.get_device_name(0)\nvram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)\nmajor, minor = torch.cuda.get_device_capability(0)\ndtype = torch.bfloat16 if major >= 8 else torch.float16\ntorch.manual_seed(SEED)\nprint(f'GPU={gpu}; VRAM_GB={vram_gb}; dtype={dtype}')\n\nstarted = time.time()\npipe = WanPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)\npipe.enable_model_cpu_offload()\nif hasattr(pipe, 'enable_vae_slicing'):\n    pipe.enable_vae_slicing()\nif hasattr(pipe, 'enable_vae_tiling'):\n    pipe.enable_vae_tiling()\nnegative = 'text, letters, subtitles, watermark, logo, blurry, low quality, distorted faces, deformed hands'\nframes = pipe(prompt=PROMPT, negative_prompt=negative, height=HEIGHT, width=WIDTH, num_frames=81, guidance_scale=5.0).frames[0]\noutput = '/kaggle/working/final.mp4'\nexport_to_video(frames, output, fps=16)\nprobe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',output], capture_output=True, text=True, check=True)\nduration = float(probe.stdout.strip())\nreport = {\n  'status': 'success', 'generation_level': 'A - genuine text-to-video',\n  'provider': 'Hugging Face Diffusers / WanPipeline', 'model': MODEL_ID,\n  'model_license': 'Apache-2.0 (model card)', 'prompt': PROMPT, 'seed': SEED,\n  'gpu': gpu, 'vram_gb': vram_gb, 'width': WIDTH, 'height': HEIGHT,\n  'frames': len(frames), 'fps': 16, 'duration_seconds': duration,\n  'elapsed_seconds': round(time.time()-started, 2), 'file': 'final.mp4'\n}\nPath('/kaggle/working/generation-report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')\nassert os.path.getsize(output) > 100_000, 'Output video is unexpectedly small.'\nprint(json.dumps(report, ensure_ascii=False, indent=2))\n